In [129]:
import torch
import torch.nn as nn
import gensim
from datasets import load_dataset
from collections import Counter

ds = load_dataset("Ayon128/Banglish-English")
print(ds['train'][12])

{'Banglish': 'ami french language sikha suru korci.', 'English': "I'm starting to learn French."}


In [130]:
train_data = ds['train']
test_data = ds['test']

english_sentences_train = []
banglish_sentences_train = []
english_sentences_test = []
banglish_sentences_test = []

for item in train_data:
    english_sentences_train.append(item['English'])
    banglish_sentences_train.append(item['Banglish'])

for item in test_data:
    english_sentences_test.append(item['English'])
    banglish_sentences_test.append(item['Banglish'])
    
english_sentences_test[:5]


['Fogging off in the afternoon',
 '"An astrologer from Howrah gave one of his clients this coral. But it\'s not real coral. Ordinary people can tell."',
 "he should've been fired months ago.",
 'While nutrition is undoubtedly a cornerstone of physical health, it is just one piece of the puzzle.',
 "I don't like coffee."]

In [131]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
def preprocess_string(s):
    # Remove all non-word characters (everything except numbers and letters)
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)
    return s

def preprocess_sentence(sentences, vocab):
    processed_sentences = []
    for s in sentences:
        indices = []
        preprocess_string(s)
        indices.append(vocab.get(SOS_TOKEN, vocab[UNK_TOKEN]))
        for word in s.split():
             indices.append(vocab.get(word.lower(), vocab[UNK_TOKEN]))
        indices.append(vocab.get(EOS_TOKEN, vocab[UNK_TOKEN]))
        processed_sentences.append(indices)
    return processed_sentences
    
def vocab_build(data):
    tokens = []
    tokens.append(PAD_TOKEN)
    tokens.append(SOS_TOKEN)
    tokens.append(EOS_TOKEN)
    tokens.append(UNK_TOKEN)
    for sentence in data:
        for word in sentence.split():
            tokens.append(word.lower())
    word_freq = Counter(tokens)
    idx = 0
    vocab = {}
    for word, _ in word_freq.items():
        vocab[word] = idx
        idx += 1    
    return vocab
    
s = ['jhow are you', 'you are who']
word_freq = vocab_build(s)
ds = Counter(word_freq)
print(word_freq)
        

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'jhow': 4, 'are': 5, 'you': 6, 'who': 7}


In [132]:


class Encoder(nn.Module):
    def __init__(self, emb_size, hidden_dim, num_layers, vocab_size, batch_size, device):
        super(Encoder, self).__init__()
        self.device = device
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.batch_size = batch_size
        self.embedding = nn.Embedding (vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size= emb_size,hidden_size = hidden_dim,num_layers=num_layers,batch_first=True)

    def forward(self, data):

        h_0 = torch.zeros(self.lstm.num_layers, self.batch_size, self.hidden_dim).to(self.device)
        c_0 = torch.zeros(self.lstm.num_layers, self.batch_size, self.hidden_dim).to(self.device)
        # print('DATA--==Encoder', data.shape)
        # print('h0---->>>', h_0.shape)
        # print('C0---->>>', c_0.shape)
        embedded_data = self.embedding(data)
        # print('Encoder input', embedded_data.shape)
        
        out, (hidden, cell) = self.lstm(embedded_data, (h_0, c_0))
        return out, hidden, cell

In [133]:
class Decoder(nn.Module):
    def __init__( self, emb_size,hidden_dim, num_layers, vocab_size, batch_size, device):
        super(Decoder, self).__init__()
        self.emb_size = emb_size
        self.device = device
        self.hidden_dim = hidden_dim
        self.batch_size = batch_size
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size=emb_size, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, data, hidden, cell):

        # print('DATA--==Decoder', data.shape)
        # if hidden.dim() == 3:
        #     hidden = hidden.squeeze(0)
        #     cell = cell.squeeze(0)

        # hidden = hidden.squeeze(0)  # Remove unnecessary dimensions
        # cell = cell.squeeze(0)
        # print('----->>',hidden.shape)
        embedded_data = self.embedding(data) 
        # print('Decoder input', embedded_data.shape)

        out, (dec_hidden, dec_cell) = self.lstm(embedded_data, (hidden, cell))  # (batch_size, seq_len, hidden_size)
        # print('after decode')
        output = self.fc(out.squeeze(1))  # (batch_size, vocab_size)
        return output, dec_hidden, dec_cell


In [134]:
from torch.utils.data import DataLoader, Dataset
import  re
from torch.nn.utils.rnn import pad_sequence

EPOCHS = 50
LEARNING_RATE = 0.001
BATCH_SIZE = 16
EMB_SIZE = 100
HIDDEN_DIM = 100
NUM_LAYERS = 1



class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, vocab_size).to(self.device)
        
        out, hidden, cell = self.encoder(source)
        input = target[:, 0]  # First token (usually <SOS>)

        for t in range(1, target_len):
            output, hidden, cell = self.decoder(input.unsqueeze(1), hidden, cell)
            outputs[:, t, :] = output
            input = output.argmax(1)  # Next input is the predicted word

        return outputs


class Seq2Seq2(nn.Module):
    def __init__(self, encoder, decoder,device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, vocab_size).to(self.device)
        out, hidden, cell = self.encoder(source)
        # print(hidden.shape)

        input = target[:, 0]
        for t in range(1, target_len):
            # print(hidden.shape)
            output, hidden, cell = self.decoder(target, hidden, cell)
            # outputs[:, t, :] = output
            # input = target[:, t] if torch.rand(1).item() < teacher_force_ratio else output.argmax(1)
        return outputs
        
        


In [135]:
banglish_vocab = vocab_build(banglish_sentences_train)
english_vocab = vocab_build(english_sentences_train)
VOCAB_SIZE_BANGLISH = len(banglish_vocab)
VOCAB_SIZE_ENGLISH = len(english_vocab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(EMB_SIZE, HIDDEN_DIM, NUM_LAYERS, VOCAB_SIZE_BANGLISH, BATCH_SIZE, device)
decoder = Decoder(EMB_SIZE, HIDDEN_DIM, NUM_LAYERS, VOCAB_SIZE_ENGLISH, BATCH_SIZE, device)




model = Seq2Seq(encoder, decoder, device)
criterion = nn.CrossEntropyLoss(ignore_index=english_vocab['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)



def train(model, data_loader, optimizer, criterion, device, epochs):
    model.to(device)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        epoch_correct = 0
        epoch_total = 0

        for source, target in data_loader:
            source, target = source.to(device), target.to(device)
            optimizer.zero_grad()

            output = model(source, target)

            # Remove last output token and align with target sequence
            output = output[:, :-1].reshape(-1, output.shape[2])  # Remove last predicted word
            target = target[:, 1:].reshape(-1)  # Shift target

            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            # Accuracy Calculation
            _, predicted = output.max(1)
            epoch_correct += (predicted == target).sum().item()
            epoch_total += target.size(0)

        epoch_accuracy = epoch_correct / epoch_total * 100
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(data_loader):.4f}, Accuracy: {epoch_accuracy:.2f}%")

    return epoch_loss / len(data_loader)


def train2(model, data_loader, optimizer, criterion, device, epochs):
    model.to(device)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        epoch_correct = 0  # To count the number of correct predictions
        epoch_total = 0    # Total number of tokens for accuracy calculation
        counter = 0;
        
        for source, target in data_loader:
            counter = counter  +1
            source, target = source.to(device), target.to(device)
            optimizer.zero_grad()
            
            # Forward pass through the model
            
            
            output = model(source, target)
            print('model output------>>>>', output.shape)
            # Ignore <SOS> token and reshape for loss calculation
            output = output[:, 1:].reshape(-1, output.shape[2])  # Ignore <SOS>, shape: (batch * seq_length, voc_size)
            target = target[:, 1:].reshape(-1)  # shape: (batch_size * seq_length)

            print('output-=====', output.shape)
            print(output.requires_grad) 
            print('target=====', target.shape)


            # Calculate loss
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            # Update epoch loss
            epoch_loss += loss.item()

            # Calculate accuracy
            _, predicted = output.max(1)  # Get the index of the max log-probability
            epoch_correct += (predicted == target).sum().item()  # Count correct predictions
            epoch_total += target.size(0)  # Total number of tokens processed

        # Print loss and accuracy for the current epoch
        epoch_accuracy = epoch_correct / epoch_total * 100
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(data_loader):.4f}, Accuracy: {epoch_accuracy:.2f}%")

    return epoch_loss / len(data_loader)

In [136]:
    
class TranslationDataset(Dataset):
    def __init__(self, source, target):
        self.source = preprocess_sentence(source, banglish_vocab)
        self.target = preprocess_sentence(target, english_vocab)
     
    def __len__(self):
        return len(self.source)

    def __getitem__(self, idx):
        return self.source[idx], self.target[idx]

def collate_fn(batch):
    source_sentences, target_sentences = zip(*batch)
    source_padded = pad_sequence([torch.tensor(seq) for seq in source_sentences], batch_first=True, padding_value=banglish_vocab[PAD_TOKEN])
    target_padded = pad_sequence([torch.tensor(seq) for seq in target_sentences], batch_first=True, padding_value=english_vocab[PAD_TOKEN])
    return source_padded, target_padded

# print('banglish_sentences_train', banglish_sentences_train[:2])
# print('english_sentences_train', english_sentences_train[:2])

train_dataset = TranslationDataset(banglish_sentences_train, english_sentences_train)

test_dataset = TranslationDataset(banglish_sentences_test, english_sentences_test)

pad_idx = 0  
data_loader = DataLoader(train_dataset, BATCH_SIZE, collate_fn=lambda batch: collate_fn(batch))  
# for batch in data_loader:
#     source_batch, target_batch = batch  # Unpack batch
#     print("Source Batch:", source_batch)
#     print("Target Batch:", target_batch)
#     break  # Stop after first batch
test_loader = DataLoader(test_dataset, BATCH_SIZE, collate_fn=lambda batch: collate_fn(batch)) 

train(model, data_loader, optimizer, criterion, device, EPOCHS)

Epoch [1/50], Loss: 7.4087, Accuracy: 3.45%
Epoch [2/50], Loss: 6.9343, Accuracy: 4.06%
Epoch [3/50], Loss: 6.7948, Accuracy: 4.21%
Epoch [4/50], Loss: 6.6615, Accuracy: 4.37%
Epoch [5/50], Loss: 6.5591, Accuracy: 4.46%
Epoch [6/50], Loss: 6.4547, Accuracy: 4.53%
Epoch [7/50], Loss: 6.3543, Accuracy: 4.59%
Epoch [8/50], Loss: 6.2548, Accuracy: 4.65%
Epoch [9/50], Loss: 6.1688, Accuracy: 4.71%
Epoch [10/50], Loss: 6.1043, Accuracy: 4.77%
Epoch [11/50], Loss: 6.0284, Accuracy: 4.83%
Epoch [12/50], Loss: 5.9652, Accuracy: 4.88%
Epoch [13/50], Loss: 5.9053, Accuracy: 4.94%
Epoch [14/50], Loss: 5.8343, Accuracy: 4.99%
Epoch [15/50], Loss: 5.7585, Accuracy: 5.04%
Epoch [16/50], Loss: 5.6958, Accuracy: 5.08%
Epoch [17/50], Loss: 5.6272, Accuracy: 5.14%
Epoch [18/50], Loss: 5.5676, Accuracy: 5.21%
Epoch [19/50], Loss: 5.4981, Accuracy: 5.26%
Epoch [20/50], Loss: 5.4415, Accuracy: 5.33%
Epoch [21/50], Loss: 5.3812, Accuracy: 5.38%
Epoch [22/50], Loss: 5.3333, Accuracy: 5.44%
Epoch [23/50], Loss

4.2432407382965085

In [137]:
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for source, target in test_loader:
            source, target = source.to(device), target.to(device)
            output = model(source, target, teacher_force_ratio=0)
            output = output.argmax(2)
            correct += (output == target).sum().item()
            total += target.numel()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

evaluate(model, test_loader)

TypeError: Seq2Seq.forward() got an unexpected keyword argument 'teacher_force_ratio'

In [ ]:
import torch

output = torch.rand((2, 4, 5)) 
print(output)

In [ ]:
cv = output
output_dropped = output[:, 1:, :]
print('----', output_dropped.reshape(-1))
# o = output_dropped.reshape(-1)
# print(o)

cv = cv[:, 1:].reshape(-1, cv.shape[2])
print(cv)

In [ ]:
target = torch.randint(0, 5, (2, 4)) 
print(target)

In [ ]:
import torch

# Simulated model output (logits) before softmax
# Shape: (batch_size, sequence_length, vocab_size)
output = torch.rand((2, 4, 5))  # Batch of 2, 4 time steps, 5 possible classes (vocab size)

# Target tensor (actual labels), each token is a class index
# Shape: (batch_size, sequence_length)
target = torch.randint(0, 5, (2, 4))  # Batch of 2, 4 time steps, class indices in range [0,4]

print("Original Output Shape:", output.shape)  # (2, 4, 5)
print("Original Target Shape:", target.shape)  # (2, 4)

# Ignore the first token (<SOS>), so we remove the first column (axis=1)
output = output[:, 1:]  # Now shape is (2, 3, 5)
target = target[:, 1:]  # Now shape is (2, 3)

print("\nAfter Removing <SOS>:")
print("Output Shape:", output.shape)  # (2, 3, 5)
print("Target Shape:", target.shape)  # (2, 3)

# Reshape for loss computation
output = output.reshape(-1, output.shape[2])  # (2*3, 5) = (6, 5)
target = target.reshape(-1)  # (2*3,) = (6,)